# Phase 6: 4B Agent Evaluation & 2B vs 4B Comparison

This notebook evaluates the scaled **Qwen3.5-4B Base Model** in the autonomous tool-use agent loop and compares its performance with the fine-tuned **Qwen3.5-2B SFT Model** on the complex dataset (`data/train_complex_500.jsonl`).

### Research Questions
1. **Scaling Impact**: Does increasing capacity from 2B to 4B parameters improve zero-shot ReAct tool compliance and code reasoning?
2. **Recovery Rate**: Can the 4B model effectively use test feedback (`run_tests`) to recover from initial failures?

In [1]:
# 1. Verify Local 4B Model Cache
import os
from pathlib import Path

cache_dir = Path("D:/huggingface_cache/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a")
if cache_dir.exists():
    print("Qwen3.5-4B snapshot verified in local cache:")
    for f in sorted(cache_dir.glob("*.safetensors")):
        print(f"  - {f.name} ({f.stat().st_size / 1e9:.2f} GB)")
else:
    print("Warning: Specific snapshot not found. Make sure Qwen/Qwen3.5-4B is downloaded in D:/huggingface_cache.")

Qwen3.5-4B snapshot verified in local cache:
  - model.safetensors-00001-of-00002.safetensors (5.33 GB)
  - model.safetensors-00002-of-00002.safetensors (3.99 GB)


### 2. Run Agent Evaluation on Qwen3.5-4B

Runs the agent loop on 20 complex tasks using `configs/agent_eval_4b.yaml`.
Results are automatically logged to the root-level directory `experiments/exp05_agent_4b/`.

In [2]:
from llm_lab.agent.evaluator import evaluate_agent

config_path = "configs/agent_eval_4b.yaml"
print(f"Starting Agent evaluation for 4B model using {config_path}...")
evaluate_agent(config_path)

d:\Github_Clones\local-llm-lab\.venv\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
d:\Github_Clones\local-llm-lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting Agent evaluation for 4B model using configs/agent_eval_4b.yaml...
Loading dataset from D:\Github_Clones\local-llm-lab\data\train_complex_500.jsonl...
Loading tokenizer Qwen/Qwen3.5-4B (cache_dir=D:/huggingface_cache)...
Loading model Qwen/Qwen3.5-4B (cache_dir=D:/huggingface_cache)...


Loading weights: 100%|██████████| 426/426 [00:03<00:00, 107.24it/s]


Running agent on 20 tasks...


100%|██████████| 20/20 [35:32<00:00, 106.65s/it]


=== Agent Evaluation ===
Task Completion Rate: 30.0%
Avg Steps to Success: 4.8
Recovery Rate:        26.3%

Saved traces to D:\Github_Clones\local-llm-lab\experiments\exp05_agent_4b/agent_traces.jsonl


### 3. Display and Compare 2B vs 4B Metrics

Loads metrics from both `experiments/exp05_agent_2b/summary.json` and `experiments/exp05_agent_4b/summary.json` and presents a side-by-side comparison.

In [3]:
import json
import pandas as pd
from llm_lab.constants import resolve_path

path_2b = resolve_path("experiments/exp05_agent_2b/summary.json")
path_4b = resolve_path("experiments/exp05_agent_4b/summary.json")

results = {}
if path_2b.exists():
    with open(path_2b, "r") as f:
        results["Qwen3.5-2B (SFT)"] = json.load(f)["metrics"]
else:
    print(f"Notice: {path_2b} not found.")

if path_4b.exists():
    with open(path_4b, "r") as f:
        results["Qwen3.5-4B (Base)"] = json.load(f)["metrics"]
else:
    print(f"Notice: {path_4b} not generated yet. Please run Cell 2 above first.")

if results:
    df = pd.DataFrame(results)
    table_data = {}
    if "task_completion_rate" in df.index:
        table_data["Task Completion Rate (%)"] = (df.loc["task_completion_rate"] * 100).round(1)
    if "recovery_rate" in df.index:
        table_data["Recovery Rate (%)"] = (df.loc["recovery_rate"] * 100).round(1)
    if "steps_to_success" in df.index:
        table_data["Avg Steps to Success"] = df.loc["steps_to_success"].round(2)
    
    summary_df = pd.DataFrame(table_data).T
    print("\n================ Agent Evaluation Metrics ================")
    display(summary_df)


================ Agent Evaluation Metrics ================


,Qwen3.5-2B (SFT),Qwen3.5-4B (Base)
Task Completion Rate (%),20.0,30.00
Recovery Rate (%),20.0,26.30
Avg Steps to Success,5.0,4.83
